In [6]:
import pandas as pd
df = pd.read_csv('nlp_customer_messsage.csv')
print(df.head())

  message_id                                            message
0     MSG001  Good morning, My phone is not working :(  Than...
1     MSG002  Hi, Thanks for your quick response! I need hel...
2     MSG003  Hi support, I LOVE this product 😍😍 Can someone...
3     MSG004  Hey there, The new update is AMAZING!! Please ...
4     MSG005  Dear team, The price is 250 EGP, too expensive...


In [7]:
df.shape

(80, 2)

In [8]:
print(df.isnull().sum())

message_id    0
message       0
dtype: int64


In [10]:
print(df['message_id'].duplicated().sum())

0


In [12]:
df['char_count'] = df['message'].astype(str).apply(len)

longest_message = df.loc[df['char_count'].idxmax()]
shortest_message = df.loc[df['char_count'].idxmin()]

print(longest_message)
print(shortest_message)

message_id                                               MSG036
message       Hey there, I need help with payment for order ...
char_count                                                  209
Name: 35, dtype: object
message_id                                            MSG029
message       Dear team, Thanks! Great service :) Thank you.
char_count                                                46
Name: 28, dtype: object


In [ ]:
import re

messages = df['message'].astype(str)

email_pattern = r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}'
df['emails'] = messages.apply(lambda x: re.findall(email_pattern, x))

url_pattern = r'https?://\S+|www\.\S+'
df['urls'] = messages.apply(lambda x: re.findall(url_pattern, x))

phone_pattern = r'\b010\d{8}\b'
df['phones'] = messages.apply(lambda x: re.findall(phone_pattern, x))

number_pattern = r'\d+'
df['numbers'] = messages.apply(lambda x: re.findall(number_pattern, x))

urgent_pattern = r'\burgent\b'
df['has_urgent'] = messages.apply(lambda x: bool(re.search(urgent_pattern, x, re.IGNORECASE)))

extracted_sample = df[(df['emails'].str.len() > 0) | 
                      (df['urls'].str.len() > 0) | 
                      (df['phones'].str.len() > 0) | 
                      (df['has_urgent'] == True)]

print("--- 5 Sample Messages with Extracted Info ---")
print(extracted_sample[['message', 'emails', 'urls', 'phones', 'numbers', 'has_urgent']].head(5))

--- 5 Sample Messages with Extracted Info ---
                                             message                   emails  \
2  Hi support, I LOVE this product 😍😍 Can someone...  [customer3@example.com]   
3  Hey there, The new update is AMAZING!! Please ...                       []   
4  Dear team, The price is 250 EGP, too expensive...                       []   
5  Dear support, The package for order 10102 is d...  [customer6@example.com]   
6  Hello, Bad quality... very disappointed. What ...                       []   

                                      urls         phones             numbers  \
2                                       []             []                 [3]   
3  [https://help.example.com/ticket/10068]             []             [10068]   
4                                       []  [01010011705]  [250, 01010011705]   
5                                       []             []          [10102, 6]   
6                                       []             []     

In [14]:
def clean_text(text):
    if not isinstance(text, str):
        return ""
        
    text = re.sub(email_pattern, '[EMAIL]', text)
    
    text = re.sub(url_pattern, '[URL]', text)
    
    text = re.sub(phone_pattern, '[PHONE]', text)
    
    text = re.sub(number_pattern, '[NUMBER]', text)
    
    text = re.sub(r'\s+', ' ', text)
    
    
    text = text.strip()
    
    return text

df['clean_message'] = df['message'].apply(clean_text)
print(df[['message', 'clean_message']].head())

                                             message  \
0  Good morning, My phone is not working :(  Than...   
1  Hi, Thanks for your quick response! I need hel...   
2  Hi support, I LOVE this product 😍😍 Can someone...   
3  Hey there, The new update is AMAZING!! Please ...   
4  Dear team, The price is 250 EGP, too expensive...   

                                       clean_message  
0  Good morning, My phone is not working :( Thank...  
1  Hi, Thanks for your quick response! I need hel...  
2  Hi support, I LOVE this product 😍😍 Can someone...  
3  Hey there, The new update is AMAZING!! Please ...  
4  Dear team, The price is [NUMBER] EGP, too expe...  


In [15]:
print("--- Sample of 8 rows ---")
print(df[['message', 'clean_message']].head(8))

--- Sample of 8 rows ---
                                             message  \
0  Good morning, My phone is not working :(  Than...   
1  Hi, Thanks for your quick response! I need hel...   
2  Hi support, I LOVE this product 😍😍 Can someone...   
3  Hey there, The new update is AMAZING!! Please ...   
4  Dear team, The price is 250 EGP, too expensive...   
5  Dear support, The package for order 10102 is d...   
6  Hello, Bad quality... very disappointed. What ...   
7  Hello team, Great quality, but delivery took 7...   

                                       clean_message  
0  Good morning, My phone is not working :( Thank...  
1  Hi, Thanks for your quick response! I need hel...  
2  Hi support, I LOVE this product 😍😍 Can someone...  
3  Hey there, The new update is AMAZING!! Please ...  
4  Dear team, The price is [NUMBER] EGP, too expe...  
5  Dear support, The package for order [NUMBER] i...  
6  Hello, Bad quality... very disappointed. What ...  
7  Hello team, Great quality, 

In [17]:
test_strings = [
    "Contact support at help@example.com for refunds.",
    "My new number is 01012345678, call me.",
    "Visit our new page https://www.website.com.",
    "This is a completely normal sentence."
]

for text in test_strings:
    print(f"Original: {text}")
    print(f"Cleaned:  {clean_text(text)}\n")

Original: Contact support at help@example.com for refunds.
Cleaned:  Contact support at [EMAIL] for refunds.

Original: My new number is 01012345678, call me.
Cleaned:  My new number is [PHONE], call me.

Original: Visit our new page https://www.website.com.
Cleaned:  Visit our new page [URL]

Original: This is a completely normal sentence.
Cleaned:  This is a completely normal sentence.

